# 01 - Dataset audit and manifest
Reports what is **actually** in the attached MetaPKLot dataset: file counts, image sizes, corrupt files, duplicates, annotation files and their structure, and hints about cameras/dates/sequences. It assumes no layout and invents nothing.

**Inputs:** the MetaPKLot dataset attached via *Add Input*. **Outputs (in `/kaggle/working/artifacts/audit/`):** `audit_summary.json`, `manifests/image_inventory.csv`, `manifests/annotation_inventory.csv`, `manifests/dataset_hashes.json`, `corrupt_report.json`, `duplicate_report.json`, `tree.txt`, `status.json`.

**Run order:** first with `SMOKE = True` (fast sample), check the output, then `SMOKE = False` for the full audit.

In [ ]:
# --- Setup: get the project code onto Kaggle (internet must be ON) ---
import os, sys, subprocess
REPO_URL = "https://github.com/Amith2277/Edge-Computing-EndSem.git"
REPO_DIR = "/kaggle/working/repo"
# If the repo is private or internet is off: attach the repo as a Kaggle dataset and set REPO_DIR to its path, e.g.
# REPO_DIR = "/kaggle/input/adaptivepark-code"
if not os.path.isdir(REPO_DIR):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
    print(r.stdout)
    if r.returncode != 0:
        raise SystemExit("git clone failed. Turn Internet ON in notebook settings, or attach the repo as a dataset.")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"])
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print("repo at", REPO_DIR)

In [ ]:
# Step 1: find the dataset. Prints the top-level of everything attached under /kaggle/input.
import os
for d in sorted(os.listdir("/kaggle/input")):
    p = os.path.join("/kaggle/input", d)
    print(d, "->", sorted(os.listdir(p))[:15])

In [ ]:
# Step 2: configuration.
SMOKE = True                       # True = audit an evenly spaced sample; set False for the full run
DATA_ROOT = "/kaggle/input/REPLACE_WITH_DATASET_SLUG"   # copy the path printed above (the folder that contains the data)
OUT = "/kaggle/working/artifacts/audit"

In [ ]:
# Step 3: run the audit.
import subprocess, sys
cmd = [sys.executable, "scripts/audit_dataset.py", "--root", DATA_ROOT, "--out", OUT] + (["--smoke"] if SMOKE else [])
r = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, universal_newlines=True)
print(r.stdout[-6000:])
assert r.returncode == 0, "audit failed, see output above"

In [ ]:
# Step 4: folder tree and annotation structure (these decide what we can evaluate).
print(open(OUT + "/tree.txt").read()[:4000])
import pandas as pd
a = pd.read_csv(OUT + "/manifests/annotation_inventory.csv")
pd.set_option("display.max_colwidth", 140, "display.width", 200)
print(a[["path", "kind", "detail"]].head(30).to_string())

## What to send back
Paste the printed `audit_summary.json`, the folder tree, and the annotation table. From them we decide: are there vehicle boxes or masks, slot polygons, occupancy labels, sequences/timestamps, and how to split.

If **no slot polygons or occupancy labels** are found, slot-level evaluation is marked **BLOCKED** (we never invent polygons).

Then set `SMOKE = False`, run again, and **Save Version** so the manifests can be attached to the next notebooks.